## Middleware in LangChain
Middleware provides a way to control and customize what happens inside an AI agent during execution. It allows developers to inspect, modify, and manage agent behavior at different stages of its workflow.
#### Key Uses of Middleware
- Logging and Monitoring: Track agent behavior, execution steps, errors, and performance.
- Prompt and Output Transformation: Modify prompts, tool selection, and output formatting.
- Retries and Fallbacks: Retry failed operations, switch to alternative models, or stop execution when necessary.
- Rate Limiting: Control how frequently models or tools can be called.
- Guardrails: Apply rules to restrict unsafe or unwanted behavior.
- PII Detection: Detect or handle personally identifiable information, such as email addresses and phone numbers.
#### How It Works
1. The agent begins execution.
2. Middleware runs at configured stages of the workflow.
3. It inspects or modifies the agent's state, model requests, or tool interactions.
4. The agent continues, retries, or terminates based on the middleware logic.



### Built-in Middleware in LangChain

- **`ModelRetryMiddleware`** — Retries failed LLM calls.
- **`ToolRetryMiddleware`** — Retries failed tool executions.
- **`ToolErrorMiddleware`** — Handles tool errors and returns error messages to the model.
- **`ModelFallbackMiddleware`** — Switches to another model if the primary model fails.
- **`SummarizationMiddleware`** — Summarizes older conversation history to manage context length.
- **`HumanInTheLoopMiddleware`** — Requests human approval before selected tool calls.
- **`ModelCallLimitMiddleware`** — Limits the number of model calls.
- **`ToolCallLimitMiddleware`** — Limits the number of tool calls.
- **`PIIMiddleware`** — Detects and handles sensitive personal information.
- **`TodoListMiddleware`** — Helps an agent plan and track tasks.
- **`LLMToolSelectorMiddleware`** — Selects relevant tools for a user request.
- **`ContextEditingMiddleware`** — Removes or trims tool results to manage context size.

In [27]:
import os
from dotenv import load_dotenv
load_dotenv(r"D:\GenAI\.env", override=True)

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")



## Summarization Middleware

Automatically summarizes conversation history when approaching token limits, preserving recent messages while compressing older context.

##### Use Cases
- Long-running conversations that exceed the context window.
- Multi-turn dialogues with extensive history.
- Applications where preserving conversation context matters.

### Message Lengths

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message based summarization
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages",10),  # summarize when length of messages becomes 10
            keep=("messages",4)       # no. of recent messages to keep as it is without summarizing
        )
    ]
)

### Run with thread_id
config = {
    "configurable":{
        "thread_id":"test-1"
    }
}

# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='d9a27949-f38b-45d4-a0c2-734f46971918'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question. We just answer 4.'}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 78, 'total_tokens': 112, 'completion_time': 0.07063505, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.004567507, 'prompt_tokens_details': None, 'queue_time': 0.349556953, 'total_time': 0.075202557}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_64b2f1c926', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1841f-6c15-70d4-a6f7-c500a2ff9e1d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 34, 'total_tokens': 112, 'output_token_details': {'reasoning': 15}})]}
Message

### Tokens

In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ~ 1 token

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [HumanMessage(content=f"Find hotels in {city}")]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city} - {tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages'][-1]}")

Paris - 164 tokens, 4 messages
content=[{'type': 'text', 'text': 'Here are some hotel options in Paris:\n\n*   **Grand Hotel:** 5-star, $350/night, featuring a spa, pool, and gym.\n*   **City Inn:** 4-star, $180/night, featuring a business center.\n*   **Budget Stay:** 3-star, $75/night, featuring free Wi-Fi.', 'extras': {'signature': 'EnMKcQFpFH0TYdXsO86BmTyyAHl8gGDNdWrvSb/Isg9QwsXx0anTKkWaHWedG77dbajlHC773crcHu7aIKmrXWZzVBDUwJJL5/sEbSseoGXvEEaxk5T1bpBdpiFi3mfNHL9XE1fUx2plqfpuD3qIt8PiDNix'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a18431-8aa5-7421-8f61-79a3d9d45528-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 148, 'output_tokens': 84, 'total_tokens': 232, 'input_token_details': {'cache_read': 0}}
London - 329 tokens, 8 messages
content=[{'type': 'text', 'text': 'Here are some hotel options in London:\n\n*   **Grand Hotel:** 5-

### Fractions

In [29]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# Low fraction for testing!
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000 
    print(f"{city}: ~{tokens} tokens (fraction: {fraction:.4%}, {len(response['messages'])} msgs)")
    print(response['messages'])

Paris: ~110 tokens (fraction: 0.0859%, 4 msgs)
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='20d0b0e2-a8ba-4c05-8b04-8dd06eb46713'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_4531717': 'EnMKcQFpFH0TUYJqPiYkcN6gxGqWDxJBgbKEbzl4WuKOMiLP0HKs8xS/QzpHDk9k+YsM0/sl8VQWVS4xRDkwSk+sU3Rx2tl+2x+uCxaXJUYIXIreFesUbO3WKQTnPzEC9bXvxbV8pOqLr7ZwatnmwuANP5qD'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a18432-75b0-7457-a90d-f5af48ab4cb4-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_4531717', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 16, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in P

## Human-in-the-Loop Middleware

Pauses agent execution for human approval, editing, or rejection of tool calls before they execute.

### Use Cases
- High-stakes operations requiring human approval (e.g., database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.